In [1]:
import pandas as pd
#First look at the data
df = pd.read_csv("fraudTest.csv", index_col="trans_num")

df.drop_duplicates(inplace=True)
df.dropna(inplace=True)

df.rename(columns= {df.columns[0]: "ID"} , inplace=True)

df.shape

(555719, 22)

In [2]:
print(df.info())

print(df.describe())

print(df.shape)

<class 'pandas.DataFrame'>
Index: 555719 entries, 2da90c7d74bd46a0caf3777415b3ebd3 to 1765bb45b3aa3224b4cdcb6e7a96cee3
Data columns (total 22 columns):
 #   Column                 Non-Null Count   Dtype  
---  ------                 --------------   -----  
 0   ID                     555719 non-null  int64  
 1   trans_date_trans_time  555719 non-null  str    
 2   cc_num                 555719 non-null  int64  
 3   merchant               555719 non-null  str    
 4   category               555719 non-null  str    
 5   amt                    555719 non-null  float64
 6   first                  555719 non-null  str    
 7   last                   555719 non-null  str    
 8   gender                 555719 non-null  str    
 9   street                 555719 non-null  str    
 10  city                   555719 non-null  str    
 11  state                  555719 non-null  str    
 12  zip                    555719 non-null  int64  
 13  lat                    555719 non-null  float64


In [3]:
df['is_fraud'] = df['is_fraud'].astype(int)
print(df['is_fraud'].value_counts())

is_fraud
0    553574
1      2145
Name: count, dtype: int64


In [4]:
X = df.select_dtypes(include=['number'])
X.drop(columns = ["is_fraud"], inplace = True)

y = df["is_fraud"]

print(X.shape)

(555719, 10)


In [5]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier

# Splitting the testing and training data 80% - 20%
X_train, X_test, y_train, y_test = train_test_split(X, y, stratify=y, test_size=0.2, random_state=10)


In [6]:
from imblearn.over_sampling import SMOTE
smote = SMOTE(random_state=10)
X_train_scaled, y_train_scaled = smote.fit_resample(X_train, y_train)

In [ ]:
classifier = RandomForestClassifier(n_estimators=100, random_state=10, max_depth=10, class_weight= 'balanced')

#Firt we fit the model to the training data. Usually the longest part.
classifier.fit(X_train_scaled, y_train_scaled)

,n_estimators,100
,criterion,'gini'
,max_depth,10
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,'sqrt'
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [8]:
y_predicted = classifier.predict(X_test)

y_probability = classifier.predict_proba(X_test)[:,1]

print(y_predicted, y_probability)
print(len(y_predicted), "                        ",len(y_probability))

[0 0 0 ... 0 0 0] [0.19208098 0.05483276 0.00025345 ... 0.08263186 0.16038032 0.13715638]
111144                          111144


In [9]:
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

print("Classification report: ")
print()
print(classification_report(y_test, y_predicted))
print()

print()
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_predicted))
print()

print()
print(f"ROC_AUC Score: {roc_auc_score(y_test, y_predicted)}")
print()

Classification report: 

              precision    recall  f1-score   support

           0       1.00      0.96      0.98    110715
           1       0.08      0.77      0.14       429

    accuracy                           0.96    111144
   macro avg       0.54      0.87      0.56    111144
weighted avg       1.00      0.96      0.98    111144



Confusion Matrix:
[[106775   3940]
 [    98    331]]


ROC_AUC Score: 0.8679874521901347



In [10]:
# Threshold tuning

from sklearn.metrics import precision_recall_curve
threshold = 0.3
y_predicted_custom = (y_probability > threshold).astype(int)

print(y_predicted_custom)

[0 0 0 ... 0 0 0]
